In [70]:
# Import necessary libraries
import pandas as pd
from pathlib import Path

In [71]:
# Pipe constants
pipe_length = 50.0
pipe_diameter = 0.2

# Sensor positions
NODE_A_X = 5.0
NODE_B_X = 25.0
NODE_C_X = 45.0
tolerance = 0.5

feature_columns = [
    "pressure",
    "velocity-magnitude",
    "turb-kinetic-energy",
    "turb-diss-rate",
    "wall-shear",
    "tailings-vof"
]

raw_data_path = Path.cwd().parent / "data" / "raw"
synthetic_path = Path.cwd().parent / "data" / "synthetic"
output_path = Path.cwd().parent / "data" / "processed"
output_path.mkdir(parents=True, exist_ok=True)

print("CONFIGURATION")
print(f"Node A: {NODE_A_X}, Node B: {NODE_B_X}, Node C: {NODE_C_X}")
print(f"Output path: {output_path}")

CONFIGURATION
Node A: 5.0, Node B: 25.0, Node C: 45.0
Output path: /home/darlenewendie/IdeaProjects/ai-pipeline-leak-detection/ml_service/data/processed


In [72]:
# Effect factor category
def get_effect_factor(name):
    name = name.lower()

    if "25" in name:
        return 0.25
    elif "50" in name:
        return 0.55
    elif "75" in name:
        return 0.90
    else:
        return 0.0

In [73]:
# Loading the three data files.
scenario_data = []

def load_csv_with_label(path, scenario_id, label):
    df = pd.read_csv(path)
    df["effect_factor"] = get_effect_factor(str(scenario_id))
    df["scenario"] = scenario_id
    df["label"] = label
    return df

# Normal
normal_file = raw_data_path / "normal_dataset.csv"
scenario_data.append((load_csv_with_label(normal_file, "normal_baseline", 0), "normal_baseline", 0))

for file in (synthetic_path / "normal_variations").glob("*.csv"):
    scenario_data.append((load_csv_with_label(file, file.stem, 0), file.stem, 0))

# Leak
for file in (synthetic_path / "leakage_variations").glob("*.csv"):
    scenario_data.append((load_csv_with_label(file, file.stem, 1), file.stem, 1))

# Blockage
for file in (synthetic_path / "blockage_variations").glob("*.csv"):
    scenario_data.append((load_csv_with_label(file, file.stem, 2), file.stem, 2))

# Summary
print("SCENARIO SUMMARY")
print("Total scenarios:", len(scenario_data))
print("Normal:", sum(1 for _,_,l in scenario_data if l==0))
print("Leak:", sum(1 for _,_,l in scenario_data if l==1))
print("Blockage:", sum(1 for _,_,l in scenario_data if l==2))

SCENARIO SUMMARY
Total scenarios: 48
Normal: 16
Leak: 16
Blockage: 16


In [74]:
# Fault type label
def get_fault_type(label):
    mapping = {
        0: "Normal",
        1: "Leak",
        2: "Blockage"
    }
    return mapping.get(label, "Unknown")

In [76]:
# Extracting sensor data points from global dataset
# Finds the closest point to our sensor nodes A,B, C
# Takes measurement from there and saves
def closest_node(df_t, x_target):
    idx = (df_t["x-coordinate"] - x_target).abs().idxmin()
    return df_t.loc[idx]

def extract_sensor_readings(df_scenario, scenario_id, label):
    rows = []

    for t, df_t in df_scenario.groupby("timestep"):

        a = closest_node(df_t, NODE_A_X)
        b = closest_node(df_t, NODE_B_X)
        c = closest_node(df_t, NODE_C_X)

        effect_factor = df_t["effect_factor"].iloc[0]
        fault_type = get_fault_type(label)

        rows.append({
            "scenario_id": scenario_id,
            "timestep": t,

            "node_a_pressure": a["pressure"],
            "velocity_a": a["velocity-magnitude"],
            "tke_a": a["turb-kinetic-energy"],
            "tdr_a": a["turb-diss-rate"],
            "wall_shear_a": a["wall-shear"],
            "tailings_vof_a": a["tailings-vof"],

            "node_b_pressure": b["pressure"],
            "velocity_b": b["velocity-magnitude"],
            "tke_b": b["turb-kinetic-energy"],
            "tdr_b": b["turb-diss-rate"],
            "wall_shear_b": b["wall-shear"],
            "tailings_vof_b": b["tailings-vof"],

            "node_c_pressure": c["pressure"],
            "velocity_c": c["velocity-magnitude"],
            "tke_c": c["turb-kinetic-energy"],
            "tdr_c": c["turb-diss-rate"],
            "wall_shear_c": c["wall-shear"],
            "tailings_vof_c": c["tailings-vof"],

            "label": label,
            "effect_factor": effect_factor,
            "fault_type": fault_type
        })
    df_out = pd.DataFrame(rows)

    print(f"[OK] {scenario_id} → shape {df_out.shape}")

    return df_out

In [77]:
def calculate_derived_features(df_wide):

    df = df_wide.copy()
    # Sort properly
    df = df.sort_values(["scenario_id", "timestep"])

    print("Calculating derived features...")
    # Pressure drops
    df["pressure_drop_ab"] = df["node_a_pressure"] - df["node_b_pressure"]
    df["pressure_drop_bc"] = df["node_b_pressure"] - df["node_c_pressure"]
    df["pressure_drop_ac"] = df["node_a_pressure"] - df["node_c_pressure"]

    # dp/dt per scenario
    def compute_group(group):
        dt = 0.5

        group["dp_dt_a"] = group["node_a_pressure"].diff().fillna(0) / dt
        group["dp_dt_b"] = group["node_b_pressure"].diff().fillna(0) / dt
        group["dp_dt_c"] = group["node_c_pressure"].diff().fillna(0) / dt

        return group

    df = df.groupby("scenario_id", group_keys=False).apply(compute_group)

    # Flow velocity (bulk proxy)
    df["flow_velocity"] = df["velocity_b"]

    # Final cleanup
    df[["dp_dt_a", "dp_dt_b", "dp_dt_c"]] = df[
        ["dp_dt_a", "dp_dt_b", "dp_dt_c"]
    ].fillna(0.0)

    # Debug output
    print("Derived features added successfully")
    print("Final shape:", df.shape)
    print("Columns added:")
    print([
        "pressure_drop_ab",
        "pressure_drop_bc",
        "pressure_drop_ac",
        "dp_dt_a",
        "dp_dt_b",
        "dp_dt_c",
        "flow_velocity"
    ])

    return df

In [78]:
print("EXTRACTING SENSOR READINGS FROM ALL SCENARIOS")
print("=" * 60)

all_extracted = []

total = len(scenario_data)

for i, (df_scenario, scenario_id, label) in enumerate(scenario_data, 1):

    print(f"\n[{i}/{total}] Processing: {scenario_id}")

    extracted = extract_sensor_readings(
        df_scenario,
        scenario_id,
        label
    )

    print(f"  Extracted shape: {extracted.shape}")

    all_extracted.append(extracted)

print("\nCalculating derived features...")

df_all = pd.concat(all_extracted, ignore_index=True)

df_all = calculate_derived_features(df_all)
# Final checks
print("\nFINAL SHAPE:", df_all.shape)

print("\nLABEL DISTRIBUTION:")
print(df_all["label"].value_counts())
# SAVE OUTPUT
output_file = raw_data_path / "sensor_point_dataset_raw.csv"
df_all.to_csv(output_file, index=False)

print("\nDATA SAVED SUCCESSFULLY")
print("Saved to:", output_file)

EXTRACTING SENSOR READINGS FROM ALL SCENARIOS

[1/48] Processing: normal_baseline
[OK] normal_baseline → shape (700, 23)
  Extracted shape: (700, 23)

[2/48] Processing: normal_v097_c103
[OK] normal_v097_c103 → shape (700, 23)
  Extracted shape: (700, 23)

[3/48] Processing: normal_v105_c095
[OK] normal_v105_c095 → shape (700, 23)
  Extracted shape: (700, 23)

[4/48] Processing: normal_v096_c104
[OK] normal_v096_c104 → shape (700, 23)
  Extracted shape: (700, 23)

[5/48] Processing: all_normal_combined
[OK] all_normal_combined → shape (700, 23)
  Extracted shape: (700, 23)

[6/48] Processing: normal_v103_c097
[OK] normal_v103_c097 → shape (700, 23)
  Extracted shape: (700, 23)

[7/48] Processing: normal_v095_c105
[OK] normal_v095_c105 → shape (700, 23)
  Extracted shape: (700, 23)

[8/48] Processing: normal_v100_c095
[OK] normal_v100_c095 → shape (700, 23)
  Extracted shape: (700, 23)

[9/48] Processing: normal_v102_c102
[OK] normal_v102_c102 → shape (700, 23)
  Extracted shape: (700, 

In [79]:
df_all.columns

Index(['timestep', 'node_a_pressure', 'velocity_a', 'tke_a', 'tdr_a',
       'wall_shear_a', 'tailings_vof_a', 'node_b_pressure', 'velocity_b',
       'tke_b', 'tdr_b', 'wall_shear_b', 'tailings_vof_b', 'node_c_pressure',
       'velocity_c', 'tke_c', 'tdr_c', 'wall_shear_c', 'tailings_vof_c',
       'label', 'effect_factor', 'fault_type', 'pressure_drop_ab',
       'pressure_drop_bc', 'pressure_drop_ac', 'dp_dt_a', 'dp_dt_b', 'dp_dt_c',
       'flow_velocity'],
      dtype='str')

In [80]:
df_all.head(20)

,timestep,node_a_pressure,velocity_a,tke_a,tdr_a,wall_shear_a,tailings_vof_a,node_b_pressure,velocity_b,tke_b,...,label,effect_factor,fault_type,pressure_drop_ab,pressure_drop_bc,pressure_drop_ac,dp_dt_a,dp_dt_b,dp_dt_c,flow_velocity
30800,1,37813.494210,2.493742,0.034635,24.489446,11.197253,0.409092,20025.090707,2.478308,0.036110,...,2,0.0,Blockage,17788.403503,15634.666083,33423.069587,0.000000,0.000000,0.000000,2.478308
30801,2,38640.203007,2.444405,0.038275,25.658411,10.641124,0.413169,20648.853362,2.425800,0.032567,...,2,0.0,Blockage,17991.349645,16054.130065,34045.479710,1653.417594,1247.525311,408.597347,2.425800
30802,3,38271.788590,2.433308,0.034279,25.294662,9.484428,0.406951,21006.247775,2.535701,0.038059,...,2,0.0,Blockage,17265.540815,17186.970665,34452.511479,-736.828834,714.788826,-1550.892373,2.535701
30803,4,37714.146140,2.437858,0.032199,27.022206,9.489620,0.401694,19696.445055,2.410991,0.038042,...,2,0.0,Blockage,18017.701086,15797.893356,33815.594442,-1115.284899,-2619.605441,158.549176,2.410991
30804,5,37854.377537,2.426621,0.037261,22.894842,10.660725,0.408602,19835.849879,2.446703,0.034532,...,2,0.0,Blockage,18018.527659,16795.110129,34813.637788,280.462794,278.809648,-1715.623898,2.446703
30805,6,38807.292051,2.529137,0.035647,23.439768,10.250242,0.406389,19166.038199,2.526261,0.035625,...,2,0.0,Blockage,19641.253852,15446.810078,35088.063930,1905.829027,-1339.623359,1356.976745,2.526261
30806,7,38164.730974,2.444688,0.034797,24.908137,10.644589,0.403550,20153.988794,2.453588,0.038813,...,2,0.0,Blockage,18010.742180,15576.550732,33587.292912,-1285.122155,1975.901189,1716.419880,2.453588
30807,8,37586.003299,2.456201,0.034091,24.837599,10.843902,0.403244,19507.289644,2.487700,0.034958,...,2,0.0,Blockage,18078.713655,16248.463098,34327.176753,-1157.455349,-1293.398300,-2637.223031,2.487700
30808,9,38078.566206,2.514600,0.033721,24.339944,9.959091,0.401573,20825.809266,2.437454,0.029658,...,2,0.0,Blockage,17252.756940,16978.248828,34231.005768,985.125815,2637.039245,1177.467786,2.437454
30809,10,38107.607361,2.471039,0.037675,25.088742,10.973236,0.410257,19831.064198,2.484224,0.033202,...,2,0.0,Blockage,18276.543163,15445.008931,33721.552094,58.082310,-1989.490136,1076.989657,2.484224
